# Imports

In [2]:
import utility
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
import torch
import torch.nn as nn
from skimage.transform import resize
from skimage.data import chelsea

In [3]:
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Macros

In [4]:
gray = cv2.cvtColor(chelsea(), cv2.COLOR_RGB2GRAY)
small_gray = gray[0:256, 0:256]

In [5]:
noise_levels = np.array([1e1, 8e0, 5e0, 3e0, 1e0, 5e-1, 1e-1])

In [6]:
wing_coefficcents = utility.coefficient_reader("ala1_181125.txt")
colessum_coefficients = utility.coefficient_reader("coliseo1_201125.txt")
usaf_coefficients = utility.coefficient_reader("usaf1_271125.txt")

In [7]:
wing = utility.weighted_hadamard_from_vector(wing_coefficcents,64)
colessum = utility.weighted_hadamard_from_vector(colessum_coefficients,64)
usaf = utility.weighted_hadamard_from_vector(usaf_coefficients,64)

# Numeric experiment

In [8]:
analysis = utility.numerical_evaluation(img=small_gray, noise_lvls=noise_levels, device=device, show=False)

In [ ]:
analysis[["noise_lvl", "ac_var", "cc_var"]]

In [ ]:
utility.plot_colored_scatter(analysis["ssim_corrupted"], analysis["ssim_denoised"],analysis["noise_lvl"], cmap="plasma", xlabel="Target SSIM", ylabel="Denoised SSIM", title="SSIM",colorbar_label="Poisson lvl", lims=True, identity_line=True)

In [ ]:
utility.plot_colored_scatter(analysis["sigma_corrupted"], analysis["sigma_denoised"], analysis["noise_lvl"], cmap="plasma", xlabel="WT MAD Noised", ylabel="WT MAD NN Denoised", title="WT MAD",colorbar_label="Poisson lvl", lims=False, identity_line=False)

In [ ]:
utility.plot_colored_scatter(analysis["alpha_corrupted"],analysis["alpha_denoised"], analysis["noise_lvl"], cmap="plasma", xlabel="2D-DFA Noised", ylabel="2D-DFA NN Denoised", title="2D DFA", lims=False, identity_line=True)

# Experimental data

In [ ]:
recon = utility.dip(img=usaf,device=device)

In [ ]:
utility.plot_comparative_histogram(usaf, recon, title1="HSPI", title2="DIP")

In [49]:
residue = usaf - recon

In [ ]:
plt.imshow(utility.image_autocorrelation(residue));

In [ ]:
plt.imshow(utility.cross_correlation_2d(recon, residue));

In [ ]:
utility.plot_dfa_with_fit(utility.dfa2d_vectorized(usaf,np.arange(6,16,1)))

In [ ]:
utility.plot_dfa_with_fit(utility.dfa2d_vectorized(recon,np.arange(6,16,1)))